In [21]:
import torch 
import torch.nn as nn
import torch.optim as optim
import math 
import torch.nn.functional as F 

class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.1):
        super().__init__()
        self.d_model = d_model
        self.d_ff = d_ff
        self.l1 = nn.Linear(d_model, d_ff)
        self.relu = nn.ReLU()
        self.l2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x):
        return self.l2(self.dropout(self.relu(self.l1(x))))

class MHA(nn.Module):
    def __init__(self, d_model, n_head, dropout=0.1):
        super().__init__()
        self.d_model = d_model 
        self.n_head = n_head 
        self.head_dim = d_model // n_head
        self.scale = self.head_dim ** -0.5
        self.qw = nn.Linear(d_model, d_model)
        self.kw = nn.Linear(d_model, d_model)
        self.vw = nn.Linear(d_model, d_model)
        self.ow = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, q, k, v, mask=None, return_attn=False):
        batch, emb = q.size(0), q.size(-1)
        ql, kl, vl = q.size(1), k.size(1), v.size(1)
        if emb != self.d_model:
            raise ValueError("emb dim error")
        q = self.qw(q).reshape(batch, ql, self.n_head, self.head_dim).transpose(1, 2)
        k = self.kw(k).reshape(batch, kl, self.n_head, self.head_dim).transpose(1, 2)
        v = self.vw(v).reshape(batch, vl, self.n_head, self.head_dim).transpose(1, 2)
        attn = torch.matmul(q, k.transpose(-2, -1)) * self.scale
        if mask is not None:
            if mask.dim() == 2:
                mask = mask.unsqueeze(0).unsqueeze(0)
            elif mask.dim() == 3:
                mask = mask.unsqueeze(1)
            if mask.dtype != torch.bool:
                mask = mask.bool()
            mask = mask.expand(batch, self.n_head, ql, kl)
            attn = attn.masked_fill(mask, float('-inf'))
        attn = F.softmax(attn, dim=-1)
        out = torch.matmul(self.dropout(attn), v).transpose(1, 2).contiguous().view(batch, ql, self.d_model)
        out = self.ow(out)
        if return_attn:
            return out, attn
        return out

class PosEncoding(nn.Module):
    def __init__(self, d_model, max_len=5_000, dropout=0.1):
        super().__init__()
        pos = torch.arange(max_len).reshape(-1, 1)
        mul = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10_000.)/d_model))
        pe = torch.zeros(max_len, d_model)
        pe[:, 0::2] = torch.sin(pos * mul)
        pe[:, 1::2] = torch.cos(pos * mul)
        self.register_buffer('pe', pe)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, x):
        return self.dropout(x + self.pe[:x.size(-2)].unsqueeze(0))

class EncoderLayer(nn.Module):
    def __init__(self, d_model, n_head, d_ff, dropout=0.1):
        super().__init__()
        self.self_attn = MHA(d_model, n_head, dropout)
        self.ffd = FeedForward(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)
        
    def forward(self, src, src_mask=None):
        src2 = self.norm1(src)
        src2 = self.self_attn(src2, src2, src2, src_mask)
        src = src + self.dropout1(src2)
        
        src2 = self.norm2(src)
        src2 = self.ffd(src2)
        src = src + self.dropout2(src2)
        return src

class Encoder(nn.Module):
    def __init__(self, d_model, n_head, d_ff, n_layer, dropout=0.1):
        super().__init__()
        self.layers = nn.ModuleList([
            EncoderLayer(d_model, n_head, d_ff, dropout)
            for _ in range(n_layer)
        ])
        self.norm = nn.LayerNorm(d_model)
    
    def forward(self, src, src_mask=None):
        output = src
        for layer in self.layers:
            output = layer(output, src_mask)
        return self.norm(output)

class DecoderLayer(nn.Module):
    def __init__(self, d_model, n_head, d_ff, dropout=0.1):
        super().__init__()
        self.self_attn = MHA(d_model, n_head, dropout)
        self.cross_attn = MHA(d_model, n_head, dropout)
        self.ffd = FeedForward(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)
        self.dropout3 = nn.Dropout(dropout)
        
    def forward(self, tgt, context, tgt_causal_mask=None, tgt_pad_mask=None, context_pad_mask=None):
        tgt2 = self.norm1(tgt)
        tgt2 = self.self_attn(tgt2, tgt2, tgt2, tgt_causal_mask)
        if tgt_pad_mask is not None:
            tgt2 = self.self_attn(tgt2, tgt2, tgt2, tgt_pad_mask)
        
        tgt = tgt + self.dropout1(tgt2)
        
        tgt2 = self.norm2(tgt)
        tgt2 = self.cross_attn(tgt2, context, context, context_pad_mask)
        tgt = tgt + self.dropout2(tgt2)
        
        tgt2 = self.norm3(tgt)
        tgt2 = self.ffd(tgt2)
        tgt = tgt + self.dropout3(tgt2)
        
        return tgt

class Decoder(nn.Module):
    def __init__(self, d_model, n_head, d_ff, n_layer, dropout=0.1):
        super().__init__()
        self.layers = nn.ModuleList([
            DecoderLayer(d_model, n_head, d_ff, dropout)
            for _ in range(n_layer)
        ])
        self.norm = nn.LayerNorm(d_model)
    
    def forward(self, tgt, context, tgt_causal_mask=None, tgt_pad_mask=None, context_pad_mask=None):
        output = tgt
        for layer in self.layers:
            output = layer(output, context, tgt_causal_mask, tgt_pad_mask, context_pad_mask)
        return self.norm(output)

class Transformer(nn.Module):
    def __init__(self, src_vocab, tgt_vocab, d_model, n_head, d_ff, n_layer, dropout=0.1, pad_idx=0, max_len=50_000):
        super().__init__()
        self.d_model = d_model
        self.pad_idx = pad_idx
        self.scale = d_model ** -0.5
        self.src_emb = nn.Embedding(src_vocab, d_model, padding_idx=pad_idx)
        self.tgt_emb = nn.Embedding(tgt_vocab, d_model, padding_idx=pad_idx)
        self.pos_encoding = PosEncoding(d_model, max_len, dropout)
        self.encoder = Encoder(d_model, n_head, d_ff, n_layer, dropout)
        self.decoder = Decoder(d_model, n_head, d_ff, n_layer, dropout)
        # logits for all vocab
        self.ow = nn.Linear(d_model, tgt_vocab)
        self._init_parameters()
        
    def _init_parameters(self):
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)
    
    def forward(self, src, tgt):
        src_pad_mask = self._make_pad_mask(src)
        tgt_pad_mask = self._make_pad_mask(tgt)
        tgt_causal_mask = self._make_causal_mask(tgt)
        
        src = self.src_emb(src) / self.scale
        src = self.pos_encoding(src)
        tgt = self.tgt_emb(tgt) / self.scale
        tgt = self.pos_encoding(tgt)
        context = self.encoder(src, src_pad_mask)
        output = self.decoder(tgt, context, tgt_causal_mask, tgt_pad_mask, src_pad_mask)
        return self.ow(output)
    
    def _make_causal_mask(self, x):
        seq = x.size(1)
        return torch.triu(torch.ones(seq, seq), diagonal=1).bool()
        
    def _make_pad_mask(self, x):
        return (x == self.pad_idx).unsqueeze(1)

In [22]:

def test_transformer_masks():
    # Test parameters
    batch_size, seq_len = 2, 10
    src_vocab_size, tgt_vocab_size = 1000, 1000
    d_model, num_heads = 512, 8
    d_ff, num_layers = 2048, 2  # Use fewer layers for testing
    pad_idx = 0
    
    # Create input with padding
    src = torch.randint(1, src_vocab_size, (batch_size, seq_len))
    tgt = torch.randint(1, tgt_vocab_size, (batch_size, seq_len))
    
    print(src.size())
    print(tgt.size())
    
    # Add padding tokens to test padding mask
    src[0, -2:] = pad_idx  # Pad last 2 positions of first batch
    tgt[1, -3:] = pad_idx  # Pad last 3 positions of second batch
    
    # Create transformer with fixed mask handling
    transformer = Transformer(
        src_vocab_size, tgt_vocab_size, d_model, num_heads,
        d_ff, num_layers, pad_idx=pad_idx
    )
    
    # Test mask creation
    src_pad_mask = transformer._make_pad_mask(src)
    tgt_pad_mask = transformer._make_pad_mask(tgt)
    tgt_causal_mask = transformer._make_causal_mask(tgt)
    
    print(f"Source padding mask shape: {src_pad_mask.shape}")
    print("Sample source padding mask:")
    print(src_pad_mask[0, 0, -5:])  # Last 5 positions of first batch
    
    print(f"\nTarget padding mask shape: {tgt_pad_mask.shape}")
    print("Sample target padding mask:")
    print(tgt_pad_mask[1, 0, -5:])  # Last 5 positions of second batch
    
    print(f"\nCausal mask shape: {tgt_causal_mask.shape}")
    print("Sample causal mask (top-left corner):")
    print(tgt_causal_mask[:5, :5])
    
    # Test forward pass with fixed masks
    output = transformer(src, tgt[:, :-1])
    print(f"\nTransformer output shape: {output.shape}")
    
    # Test that the output is valid (no NaNs)
    print("Output contains NaN:", torch.isnan(output).any().item())
    
    print("\nTransformer with fixed masks tested successfully!")

# Execute test
test_transformer_masks()

torch.Size([2, 10])
torch.Size([2, 10])
Source padding mask shape: torch.Size([2, 1, 10])
Sample source padding mask:
tensor([False, False, False,  True,  True])

Target padding mask shape: torch.Size([2, 1, 10])
Sample target padding mask:
tensor([False, False,  True,  True,  True])

Causal mask shape: torch.Size([10, 10])
Sample causal mask (top-left corner):
tensor([[False,  True,  True,  True,  True],
        [False, False,  True,  True,  True],
        [False, False, False,  True,  True],
        [False, False, False, False,  True],
        [False, False, False, False, False]])

Transformer output shape: torch.Size([2, 9, 1000])
Output contains NaN: False

Transformer with fixed masks tested successfully!
